Previous notebook showed that the CATE estimates I was getting were basically 0. 

In this notebook I've going to redo the same analysis but aplly corrections to the basefile that help focus the causal questions that we have asked.

Most notably redefining the basefile structure to be area hour rather than station hour: The formal statement is that the potential outcome for unit i depends only on unit i's own treatment, not on the treatment assigned to any other unit. Written precisely: Y_i(t_1, ..., t_n) = Y_i(t_i) for all possible treatment vectors. Violation of this is what Sobel (2006) and others call spillover or interference, and it is essentially guaranteed in your setting.
The causal mechanism is modal substitution. A strike on, say, the Central Line means thousands of commuters who would normally travel from Holborn to Liverpool Street underground are now standing on the street looking for alternatives. Some take buses, some walk, some use Santander Bikes. The demand shock at a bike station near Holborn is not caused by that station's own strike_exposed value — it is caused partly by the aggregated treatment status of all nearby stations and all nearby tube lines. Station A's potential outcome under control Y_A(0) is not the same whether station B next to it is treated or not. Your model assumes it is.
This creates two concrete problems for your CATE estimates. First, your control group is contaminated. On a strike day, stations coded as strike_exposed = 0 — those not directly adjacent to a striking line — may still see elevated ridership because displaced commuters spread out across a wider area. If those control observations have inflated Y values, the counterfactual comparison is biased downward, which would attenuate your estimated treatment effect toward zero. This is consistent with the S-Learner's near-zero ATE. Second, your standard errors are underestimated. The asymptotic normality proofs underlying CausalForestDML (Wager & Athey, 2018) assume independent potential outcomes. Spatial clustering of treatment effects violates this, so your confidence intervals will be too narrow. (from Claude)

And also to redefine strike_effected as not a binary treatment but continous, based on the number of tubes striking.

Will also look at stratified sampling to remove superflous control rows

In [16]:
import pandas as pd
import h3
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML
from xgboost import XGBRegressor, XGBClassifier

e:\tfl_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
#bring in data
PATH = "bike_hourly_with_covariates_v2_filtered.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'is_school_holiday', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
       'wind_speed_10m', 'weather_code', 'dist_nearest_tube_km',
       'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score'],
      dtype='str')

In [28]:
df[['lat', 'lon']].head()

,lat,lon
0,51.526377,-0.07813
1,51.526377,-0.07813
2,51.526377,-0.07813
3,51.526377,-0.07813
4,51.526377,-0.07813


In [29]:
def assign_h3_cell(lat, lon, resolution=8):
    # Resolution 8 gives ~460m edge hexagons, roughly a 10-min walk radius
    return h3.latlng_to_cell(lat, lon, resolution)

df["h3_cell"] = df.apply(lambda r: assign_h3_cell(r["lat"], r["lon"]), axis=1)

In [30]:
df[["h3_cell"]].head()

,h3_cell
0,88194ad347fffff
1,88194ad347fffff
2,88194ad347fffff
3,88194ad347fffff
4,88194ad347fffff


In [32]:
df.columns

Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'is_school_holiday', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
       'wind_speed_10m', 'weather_code', 'dist_nearest_tube_km',
       'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score',
       'h3_cell'],
      dtype='str')

In [36]:
# You cannot directly observe supply constraints from trips_start alone,
# but you can proxy for them using the ratio of bikes taken to time window.
# A station that records its maximum-ever hourly trips is likely supply-constrained.

# Compute per-station maximum historical hourly trips (from your original data)
# before aggregating to cell level
station_max_hourly = (
    df.groupby("station_id")["ts"]
    .max()
    .rename("station_max_hourly_trips")
    .reset_index()
)

df = df.merge(station_max_hourly, on="station_id", how="left")

# Flag observations where trips are near the historical maximum
# (likely supply-constrained)
df["near_capacity"] = (df["ts"] >= df["station_max_hourly_trips"] * 0.9).astype(int)

# Aggregate to cell level — fraction of stations near capacity
# If this is high on strike days, supply constraint is your problem
cell_capacity = (
    df.groupby(["h3_cell", "trips_start"])
    .agg(frac_near_capacity=("near_capacity", "mean"))
    .reset_index()
)

cell_hour = cell_hour.merge(cell_capacity, on=["h3_cell", "trips_start"], how="left")

# Check: are strike-exposed cells more likely to be near capacity?
print(cell_hour.groupby("frac_exposed")["frac_near_capacity"].mean())

frac_exposed
0.000000    0.000190
0.100000    0.000000
0.111111    0.000000
0.125000    0.000000
0.142857    0.000000
0.166667    0.001208
0.181818    0.000000
0.200000    0.000000
0.222222    0.000000
0.250000    0.000265
0.272727    0.000000
0.285714    0.001091
0.300000    0.000000
0.333333    0.000421
0.375000    0.000000
0.400000    0.000000
0.428571    0.000000
0.444444    0.000000
0.500000    0.000698
0.555556    0.000000
0.571429    0.000000
0.600000    0.000000
0.625000    0.000000
0.666667    0.000000
0.700000    0.002273
0.714286    0.000000
0.750000    0.001147
0.777778    0.004630
0.800000    0.000000
0.818182    0.000000
0.833333    0.000728
0.857143    0.000000
0.875000    0.005208
0.888889    0.000000
0.900000    0.000000
0.909091    0.001818
0.916667    0.000000
1.000000    0.000866
Name: frac_near_capacity, dtype: float64


In [38]:
# Step 2: Create the missing outcome columns
# Adjust the column names to match whatever your aggregation produced

# If your aggregation output column is called "total_trips":
cell_hour["y_log1p"] = np.log1p(cell_hour["total_trips"])
cell_hour["y_per_station_log1p"] = np.log1p(
    cell_hour["total_trips"] / cell_hour["n_stations"]
)

# If it's called something else (e.g. "ts" or "trips_start"):
# cell_hour["y_log1p"] = np.log1p(cell_hour["ts"])
# cell_hour["y_per_station_log1p"] = np.log1p(cell_hour["ts"] / cell_hour["n_stations"])

# Quick sanity check before re-running diagnostics
print(cell_hour[["total_trips", "n_stations", "y_log1p", "y_per_station_log1p"]].describe())
print("\nAny NaNs:", cell_hour[["y_log1p", "y_per_station_log1p"]].isna().sum().to_dict())

        total_trips    n_stations       y_log1p  y_per_station_log1p
count  2.487918e+06  2.487918e+06  2.487918e+06         2.487918e+06
mean   1.047163e+01  3.152981e+00  1.939356e+00         1.198403e+00
std    1.598175e+01  2.216496e+00  9.478790e-01         4.719305e-01
min    1.000000e+00  1.000000e+00  6.931472e-01         6.931472e-01
25%    2.000000e+00  1.000000e+00  1.098612e+00         6.931472e-01
50%    5.000000e+00  3.000000e+00  1.791759e+00         1.098612e+00
75%    1.300000e+01  4.000000e+00  2.639057e+00         1.455287e+00
max    4.280000e+02  1.200000e+01  6.061457e+00         4.779123e+00

Any NaNs: {'y_log1p': 0, 'y_per_station_log1p': 0}


In [41]:
# Priority 1: Is the naive difference already negative in raw data?
treated_mean = cell_hour.loc[cell_hour["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_hour.loc[cell_hour["frac_exposed"] == 0, "y_log1p"].mean()
print(f"Naive difference: {treated_mean - control_mean:.4f}")

# Priority 2: Is n_stations confounding the outcome?
print(cell_hour.groupby(
    pd.cut(cell_hour["frac_exposed"], bins=[-0.01, 0, 0.5, 1.01],
           labels=["control", "partial", "fully_treated"])
)[["n_stations", "y_log1p", "y_per_station_log1p"]].mean())

# Priority 3: Is n_stations actually in your feature matrix?
print("n_stations in num_cols:", "n_stations" in num_cols)

# Priority 4: Treatment rate by hour — are night hours diluting the signal?
print(cell_hour.groupby("trips_start")["frac_exposed"].mean().round(4))

Naive difference: 0.2765
               n_stations   y_log1p  y_per_station_log1p
frac_exposed                                            
control          3.147291  1.937254             1.197811
partial          4.872033  2.601125             1.374301
fully_treated    3.644144  2.113956             1.250326
n_stations in num_cols: False
trips_start
2016-01-10 00:00:00    0.0
2016-01-10 01:00:00    0.0
2016-01-10 02:00:00    0.0
2016-01-10 03:00:00    0.0
2016-01-10 04:00:00    0.0
                      ... 
2019-01-01 19:00:00    0.0
2019-01-01 20:00:00    0.0
2019-01-01 21:00:00    0.0
2019-01-01 22:00:00    0.0
2019-01-01 23:00:00    0.0
Name: frac_exposed, Length: 25941, dtype: float64


In [43]:

# Aggregate to cell-hour level
cell_hour = (
    df.groupby(["h3_cell", "trips_start"])
    .agg(
        total_trips     = ("ts", "sum"),
        frac_exposed    = ("strike_exposed", "mean"),  # continuous treatment
        n_stations      = ("station_id", "nunique"),

        # Keep covariates by averaging within cell
        temperature_2m  = ("temperature_2m", "mean"),
        relative_humidity_2m = ("relative_humidity_2m", "mean"),
        precipitation   = ("precipitation", "mean"),
        rain            = ("rain", "mean"),
        cloud_cover     = ("cloud_cover", "mean"),
        wind_speed_10m = ("wind_speed_10m", "mean"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "mean"),
        hour = ("hour", "first"),  # hour of day is constant within cell-hour
        is_am_peak      = ("is_am_peak", "first"),
        is_pm_peak      = ("is_pm_peak", "first"),
        is_school_holiday = ("is_school_holiday", "first"),
        is_bank_holiday = ("is_bank_holiday", "first"),
        dist_nearest_tube_km = ("dist_nearest_tube_km", "mean"),
        days_since_last_strike = ("days_since_last_strike", "mean"),
        cycle_infra_score = ("cycle_infra_score", "mean"),
        days_to_next_strike = ("days_to_next_strike", "mean")

        # ... other covariates
    )
    .reset_index()
)


cell_hour["y_log1p"] = np.log1p(cell_hour["total_trips"])
# Treatment is now continuous: fraction of stations in the cell that are strike-exposed
# This also addresses the hidden versions of treatment problem (see Consistency below)

In [44]:
cell_hour.shape

(2487918, 22)

In [45]:
cell_hour.head()

,h3_cell,trips_start,total_trips,frac_exposed,n_stations,temperature_2m,relative_humidity_2m,precipitation,rain,cloud_cover,...,hour,is_am_peak,is_pm_peak,is_school_holiday,is_bank_holiday,dist_nearest_tube_km,days_since_last_strike,cycle_infra_score,days_to_next_strike,y_log1p
0,88194ad101fffff,2016-01-10 02:00:00,1,0.0,1,6.3,85.0,0.0,0.0,81.0,...,2,0,0,0,0,1.598242,30.0,65.0,30.0,0.693147
1,88194ad101fffff,2016-01-10 06:00:00,2,0.0,2,5.4,80.0,0.0,0.0,100.0,...,6,0,0,0,0,1.577844,30.0,63.0,30.0,1.098612
2,88194ad101fffff,2016-01-10 08:00:00,2,0.0,1,5.7,83.0,0.0,0.0,99.0,...,8,1,0,0,0,1.557445,30.0,61.0,30.0,1.098612
3,88194ad101fffff,2016-01-10 09:00:00,1,0.0,1,5.9,84.0,0.0,0.0,98.0,...,9,1,0,0,0,1.557445,30.0,61.0,30.0,0.693147
4,88194ad101fffff,2016-01-10 10:00:00,2,0.0,2,6.3,83.0,0.0,0.0,78.0,...,10,0,0,0,0,1.645024,30.0,59.0,30.0,1.098612


In [46]:
COL_TIME = "trips_start"
COL_Y    = "total_trips"
COL_T    = "frac_exposed"

cell_hour[COL_TIME] = pd.to_datetime(cell_hour[COL_TIME])
cell_hour[COL_Y] = pd.to_numeric(cell_hour[COL_Y], errors="coerce")
cell_hour[COL_T] = pd.to_numeric(cell_hour[COL_T], errors="coerce").fillna(0).astype(int)

cell_hour = cell_hour.dropna(subset=[COL_TIME, COL_Y])
cell_hour["y_log1p"] = np.log1p(cell_hour[COL_Y].astype(float))

cell_hour[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,trips_start,total_trips,frac_exposed,y_log1p
0,2016-01-10 02:00:00,1,0,0.693147
1,2016-01-10 06:00:00,2,0,1.098612
2,2016-01-10 08:00:00,2,0,1.098612
3,2016-01-10 09:00:00,1,0,0.693147
4,2016-01-10 10:00:00,2,0,1.098612


In [47]:
SPLIT_DATE = "2018-01-01"

train = cell_hour[cell_hour[COL_TIME] < SPLIT_DATE].copy()
test  = cell_hour[cell_hour[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (1646656, 22) test: (841262, 22)


In [48]:
# Candidate numeric features (only keep those that exist)
num_cols = [
    "hour", 
    "dow",
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_am_peak",
    "is_pm_peak",
    "is_bank_holiday",
    "lat",
    "lon",
    'is_school_holiday',
    'dist_nearest_tube_km',
    'n_tube_within_500m', 
    'n_tube_within_1km', 
    'strike_severity_daily_frac',
    'days_to_next_strike',
    'days_since_last_strike',
    'cycle_infra_score',
    'n_stations'
]


#cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

#print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

num_cols: ['hour', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_am_peak', 'is_pm_peak', 'is_bank_holiday', 'is_school_holiday', 'dist_nearest_tube_km', 'strike_severity_daily_frac', 'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score', 'n_stations']
Xtr: (1646656, 17) Xte: (841262, 17)


In [49]:
print("Treatment rate:", T_train.mean())
print("Treated count:", T_train.sum(), "of", len(T_train))

Treatment rate: 0.0047429457032920055
Treated count: 7810 of 1646656


In [50]:
rf_y = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

rf_t = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

In [51]:
xg_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

xg_t = XGBClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)


Treated: 7,810  |  Control: 1,638,846  |  scale_pos_weight: 209.8


In [ ]:
import time

start = time.time()

s = SLearner(overall_model=xg_y)
s.fit(Y_train, T_train, X=Xtr)
tau_s_learner = s.effect(Xte)

s_learner_end = time.time() - start

print("SLearner done in", s_learner_end / 60, "minutes")

tlearner_start = time.time()

t = TLearner(models=xg_y
)
t.fit(Y_train, T_train, X=Xtr)
tau_t_learner = t.effect(Xte)

t_learner_end = time.time() - tlearner_start

print("TLearner done in", t_learner_end / 60, "minutes")

Xlearner_start = time.time()

x = XLearner(models=xg_y,
             propensity_model=xg_t)

x.fit(Y_train, T_train, X=Xtr)
tau_x_learner = x.effect(Xte)

xlearner_end = time.time() - Xlearner_start

print("XLearner done in", xlearner_end / 60, "minutes")

# Map each learner name to its corresponding tau array.
tau_map = {
    "s_learner": tau_s_learner,
    "t_learner": tau_t_learner,
    "x_learner": tau_x_learner,
}

att_mask = T_test > 0

for model_name, tau_values in tau_map.items():
    ate = np.mean(tau_values)
    att = np.mean(tau_values[att_mask])
    atc = np.mean(tau_values[~att_mask])

    print(f"{model_name.upper()} ATE: {np.expm1(ate)*100:.2f}%")
    print(f"{model_name.upper()} ATT: {np.expm1(att)*100:.2f}%  (effect on actually treated cells)")
    print(f"{model_name.upper()} ATC: {np.expm1(atc)*100:.2f}%  (effect if controls were treated)")


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


SLearner done in 1.0797304193178812 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TLearner done in 0.9988520026206971 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XLearner done in 2.7760329365730287 minutes
s-learner_ATE : -3.80%
s-learner_ATT : -4.98%  (effect on actually strike-exposed cells)
s-learner_ATC : -3.79%  (effect if controls had been treated)
t-learner_ATE : -3.80%
t-learner_ATT : -4.98%  (effect on actually strike-exposed cells)
t-learner_ATC : -3.79%  (effect if controls had been treated)
x-learner_ATE : -3.80%
x-learner_ATT : -4.98%  (effect on actually strike-exposed cells)
x-learner_ATC : -3.79%  (effect if controls had been treated)


In [ ]:
# Assuming your cell-hour dataframe is called cell_hour
# and has columns: frac_exposed (or strike_exposed), y_log1p, n_stations, h3_cell, trips_start

# 1. Basic treated vs control comparison
treated_mean = cell_hour.loc[cell_hour["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_hour.loc[cell_hour["frac_exposed"] == 0, "y_log1p"].mean()
naive_diff   = treated_mean - control_mean

print(f"Mean Y (treated cells) : {treated_mean:.4f}")
print(f"Mean Y (control cells) : {control_mean:.4f}")
print(f"Naive difference       : {naive_diff:.4f}")

# 2. Check n_stations distribution across treated vs control
print("\nStations per cell (treated vs control):")
print(cell_hour.groupby("frac_exposed")["n_stations"].describe())

# 3. Check how many cells are ever treated
ever_treated = cell_hour.groupby("h3_cell")["frac_exposed"].max()
print(f"\nCells ever treated  : {(ever_treated > 0).sum()}")
print(f"Cells never treated : {(ever_treated == 0).sum()}")
print(f"Fraction treated    : {(ever_treated > 0).mean():.3f}")

# 4. Check the outcome distribution
print("\nY distribution:")
print(cell_hour.groupby("frac_exposed")["y_log1p"].describe())

# 5. Time of day — are you including night hours in treatment?
print("\nTreatment rate by hour:")# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

# ── Nuisance model for Y (outcome) ───────────────────────────────────────────
# Standard regression — no imbalance issue here.
# n_estimators deliberately modest; the causal forest adds further complexity.
xgb_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Nuisance model for T (propensity) ────────────────────────────────────────
# Binary target (0/1) fit as regression — outputs E[T|X] = P(T=1|X).
# scale_pos_weight is the key addition vs. Random Forest.
xgb_t = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── CausalForestDML ───────────────────────────────────────────────────────────
cf_xgb = CausalForestDML(
    model_y          = xgb_y,
    model_t          = xgb_t,
    n_estimators     = 200,       # more trees → smoother CATE surface
    min_samples_leaf = 50,
    max_features     = "auto",
    n_jobs           = -1,
    random_state     = 0,
)

cf_xgb.fit(Y_train, T_train, X=Xtr)
tau_cf_xgb   = cf_xgb.effect(Xte)
lb_xgb, ub_xgb = cf_xgb.effect_interval(Xte, alpha=0.05)

print(f"\nATE (XGB nuisance): {np.mean(tau_cf_xgb):.6f}")
print(f"95% CI:            [{np.mean(lb_xgb):.6f}, {np.mean(ub_xgb):.6f}]")
print(cell_hour.groupby("hour")["strike_exposed"].mean().round(4))

Mean Y (treated cells) : 1.9693
Mean Y (control cells) : 1.9392
Naive difference       : 0.0301

Stations per cell (treated vs control):
                  count      mean       std  min  25%  50%  75%   max
frac_exposed                                                         
0             2475438.0  3.152608  2.215590  1.0  1.0  3.0  4.0  12.0
1               12480.0  3.227003  2.388467  1.0  1.0  3.0  4.0  12.0

Cells ever treated  : 128
Cells never treated : 4
Fraction treated    : 0.970

Y distribution:
                  count      mean       std       min       25%       50%  \
frac_exposed                                                                
0             2475438.0  1.939205  0.947411  0.693147  1.098612  1.791759   
1               12480.0  1.969262  1.036224  0.693147  1.098612  1.791759   

                   75%       max  
frac_exposed                      
0             2.639057  6.061457  
1             2.708050  5.880533  

Treatment rate by hour:
Treated: 7,81